# Parameterized Workflow Translation

## Problem

Circuit migration is more useful when it carries the execution workflow too: symbolic parameters, parameter bindings, measurement requests, shot counts, result shapes, and observable grouping.

## Translation Scope

This workflow uses `workflow-json` as the precise neutral contract, translates it to Qiskit Aer, Cirq, PennyLane, and Braket LocalSimulator source, verifies each generated source canonically, normalizes SDK-shaped result payloads, and groups Pauli terms for measurement reuse.

## Variables and Parameters

- `workflow_source`: neutral `workflow-json` source for the parameterized circuit and execution request.
- `from_format`: source workflow format, here `workflow-json` for the first translation and `qiskit` for the reimport example.
- `to_format`: target local SDK workflow source format.
- `TARGETS`: Qiskit Aer, Cirq, PennyLane, and Braket LocalSimulator workflow outputs.
- `verify`: semantic verification mode, here canonical workflow verification.
- `result_cases`: SDK-shaped result JSON fixtures normalized into portable result objects.
- `ARTIFACT_DIR`: notebook artifact directory for generated source, reports, result JSON, and grouping JSON.


## Setup

Use the public workflow translation helpers and the shared notebook artifact directory convention used by the other tutorials.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from quantum_backend_bench.core.observable_translate import translation_capability_rows
from quantum_backend_bench.core.workflow_translate import (
    WORKFLOW_OUTPUT_FORMATS,
    group_pauli_terms_source,
    import_workflow_source,
    normalize_result_source,
    translate_workflow_source,
    verify_workflow_translation,
    workflow_translation_report,
)
from quantum_backend_bench.utils.notebook import notebook_artifact_dir, verification_frame

ARTIFACT_DIR = notebook_artifact_dir()
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "examples" / "translation").exists():
    REPO_ROOT = REPO_ROOT.parent

EXAMPLE_DIR = REPO_ROOT / "examples" / "translation"
WORKFLOW_PATH = EXAMPLE_DIR / "parameterized_workflow.json"
TARGETS = ["qiskit_aer", "cirq", "pennylane", "braket_local"]


def source_preview(title, source, max_lines=34):
    lines = source.strip().splitlines()
    print(title)
    print("-" * len(title))
    for number, line in enumerate(lines[:max_lines], start=1):
        print(f"{number:>2}: {line}")
    if len(lines) > max_lines:
        print(f"... {len(lines) - max_lines} more lines")


def result_summary(results):
    rows = []
    for target, result in results.items():
        verification = result.verification
        rows.append(
            {
                "target": target,
                "verified": verification.passed if verification else None,
                "mode": verification.mode if verification else None,
                "source_lines": len(result.source.splitlines()),
                "notes": ", ".join(result.notes),
            }
        )
    return pd.DataFrame(rows)

## Source Workflow

Start from the checked-in workflow fixture. It defines a two-qubit parameterized circuit, a numeric binding for `theta`, counts/probability requests, a Pauli expectation request, and a local shot count.

In [ ]:
workflow_source = WORKFLOW_PATH.read_text(encoding="utf-8")
source_preview("Input workflow-json", workflow_source)

## Preflight Inspection

Import the workflow into the neutral model before generating SDK code. This makes the migration contract explicit and independent of any one SDK.

In [ ]:
workflow, detected_format = import_workflow_source(workflow_source, from_format="workflow-json")

preflight_rows = [
    {"field": "input format", "value": detected_format},
    {"field": "qubits", "value": workflow.n_qubits},
    {"field": "parameters", "value": ", ".join(workflow.parameters)},
    {"field": "bindings", "value": workflow.parameter_bindings},
    {"field": "operations", "value": len(workflow.operations)},
    {
        "field": "measurements",
        "value": ", ".join(request.kind for request in workflow.measurements),
    },
    {"field": "shots", "value": workflow.shots},
    {"field": "supported outputs", "value": ", ".join(WORKFLOW_OUTPUT_FORMATS)},
]
display(pd.DataFrame(preflight_rows))

## Translate to All Local SDK Targets

Each target receives a local execution wrapper, neutral result JSON construction, and an embedded `workflow_spec` block that supports canonical reimport verification.

In [ ]:
workflow_results = {
    target: translate_workflow_source(
        workflow_source,
        from_format="workflow-json",
        to_format=target,
        verify="canonical",
    )
    for target in TARGETS
}

display(result_summary(workflow_results))

## Translated Source Previews

The SDKs express the same workflow with different parameter APIs: Qiskit `Parameter`, Cirq `sympy.Symbol`, PennyLane QNode arguments, and Braket `FreeParameter`.

In [ ]:
for target, result in workflow_results.items():
    source_preview(f"{target} workflow source", result.source)
    print()

## Save Artifacts

Write one generated source file per target and one combined JSON report for migration auditing.


In [ ]:
artifact_rows = []
combined_report = []
for target, result in workflow_results.items():
    source_path = ARTIFACT_DIR / f"parameterized_workflow_to_{target}.py"
    source_path.write_text(result.source, encoding="utf-8")
    combined_report.append(
        workflow_translation_report(
            result,
            source_path=str(WORKFLOW_PATH),
            from_format="workflow-json",
            to_format=target,
        )
    )
    artifact_rows.append(
        {"artifact": "translated workflow", "target": target, "path": str(source_path)}
    )

report_path = ARTIFACT_DIR / "parameterized_workflow_all_targets_report.json"
report_path.write_text(
    json.dumps(combined_report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
artifact_rows.append({"artifact": "combined report", "target": "all", "path": str(report_path)})

display(pd.DataFrame(artifact_rows))

## Translation Audit

The audit table makes the current workflow translation surface explicit: parameterized circuits, bindings, measurement requests, execution wrappers, neutral result objects, and Pauli measurement grouping.


In [ ]:
audit_rows = []
for row in translation_capability_rows():
    audit_rows.append(
        {
            "sdk": row["sdk"],
            "parameterized circuits": row["parameterized_circuits"],
            "bindings": row["parameter_bindings"],
            "measurements": row["measurement_requests"],
            "execution": row["execution_wrappers"],
            "results": row["result_objects"],
            "grouping": row["measurement_grouping"],
        }
    )

display(pd.DataFrame(audit_rows))

## Canonical Verification

Canonical verification reimports generated SDK source and compares neutral workflow semantics: parameters, bindings, operations, measurement requests, shot count, and embedded observables.


In [ ]:
verification_rows = []
for target, result in workflow_results.items():
    verification = verify_workflow_translation(workflow, result.source, to_format=target)
    verification_rows.append(
        {
            "check": f"{target} canonical workflow verification",
            "value": verification.details,
            "expected": "passed",
            "passed": verification.passed,
        }
    )

display(verification_frame(verification_rows))

## Static SDK Reimport

Generated SDK scripts embed `workflow_spec`, so they can be used as inputs for another translation step. The same importer also supports a small static parameterized subset for common user-authored SDK snippets.


In [ ]:
qiskit_source = workflow_results["qiskit_aer"].source
qiskit_to_cirq = translate_workflow_source(
    qiskit_source,
    from_format="qiskit",
    to_format="cirq",
    verify="canonical",
)

source_preview("Qiskit workflow reimported and translated to Cirq", qiskit_to_cirq.source)
print(qiskit_to_cirq.verification.details)

## Result Normalization

`translate-result` accepts SDK-shaped JSON fixtures and normalizes them to the same portable result object used by generated workflow scripts.

In [ ]:
result_cases = [
    ("qiskit-counts-json", EXAMPLE_DIR / "qiskit_counts_result.json"),
    ("cirq-counts-json", EXAMPLE_DIR / "cirq_counts_result.json"),
    ("pennylane-samples-json", EXAMPLE_DIR / "pennylane_samples_result.json"),
    ("braket-counts-json", EXAMPLE_DIR / "braket_counts_result.json"),
]

normalized_rows = []
for from_format, path in result_cases:
    result = normalize_result_source(path.read_text(encoding="utf-8"), from_format=from_format)
    payload = json.loads(result.source)
    normalized_path = ARTIFACT_DIR / f"{path.stem}_neutral_result.json"
    normalized_path.write_text(result.source, encoding="utf-8")
    normalized_rows.append(
        {
            "from_format": from_format,
            "shots": payload["shots"],
            "states": ", ".join(sorted(payload["counts"])),
            "probability_sum": round(sum(payload["probabilities"].values()), 12),
            "path": str(normalized_path),
        }
    )

display(pd.DataFrame(normalized_rows))

## Result Probability Plot

Plot the normalized probabilities from each SDK-shaped result fixture. This checks the result-object layer visually without requiring any optional SDK runtime in the notebook environment.


In [ ]:
probability_rows = []
for from_format, path in result_cases:
    payload = json.loads(
        normalize_result_source(path.read_text(encoding="utf-8"), from_format=from_format).source
    )
    for state, probability in payload["probabilities"].items():
        probability_rows.append(
            {
                "source": from_format.replace("-json", ""),
                "state": f"|{state}>",
                "probability": probability,
            }
        )

probability_frame = pd.DataFrame(probability_rows)
probability_pivot = probability_frame.pivot_table(
    index="state", columns="source", values="probability", fill_value=0.0
).sort_index()

fig, ax = plt.subplots(figsize=(8, 3.8))
probability_pivot.plot(kind="bar", ax=ax)
ax.set_title("Normalized result probabilities")
ax.set_xlabel("state")
ax.set_ylabel("probability")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## Pauli Measurement Grouping

The expectation request in this workflow uses Pauli terms. Larger Hamiltonians can be grouped into qubit-wise commuting sets before measurement planning.

In [ ]:
grouping_source = (EXAMPLE_DIR / "ising_hamiltonian.json").read_text(encoding="utf-8")
grouping_result = group_pauli_terms_source(
    grouping_source,
    from_format="pauli-json",
    strategy="qubit-wise",
)
grouping_payload = json.loads(grouping_result.source)

grouping_path = ARTIFACT_DIR / "ising_measurement_groups.json"
grouping_path.write_text(grouping_result.source, encoding="utf-8")

display(
    pd.DataFrame(
        [
            {
                "group": group["index"],
                "terms": group["term_count"],
                "canonical": group["canonical"],
            }
            for group in grouping_payload["groups"]
        ]
    )
)
print(f"Saved grouping artifact to {grouping_path}")

## Verification Summary

All workflow translation targets should pass canonical verification. Result normalization should preserve shot totals and probabilities, and Pauli grouping should produce at least one measurement group.


In [ ]:
summary_checks = list(verification_rows)
summary_checks.extend(
    {
        "check": f"{row['from_format']} probability sum",
        "value": row["probability_sum"],
        "expected": 1.0,
        "passed": row["probability_sum"] == 1.0,
    }
    for row in normalized_rows
)
summary_checks.append(
    {
        "check": "Pauli grouping produced groups",
        "value": grouping_payload["group_count"],
        "expected": ">= 1",
        "passed": grouping_payload["group_count"] >= 1,
    }
)

display(verification_frame(summary_checks))